<a href="https://colab.research.google.com/github/sandeepmohamed666/ICTAK_B9_Python-Case-Study/blob/main/Data_Acquisition_Case_study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q. Ingest SpaceX launch and rocket data from APIs, clean and merge the data, and   store it in a local SQLite3 database. Use SQL queries to analyze the launch data.

api.spacexdata.com/v4/launches          
api.spacexdata.com/v4/rockets

You are tasked with analyzing real-time data from the SpaceX API and enhancing it with simulated attributes to understand how real-world datasets are built and analyzed using Python and SQL.

## Step 1: Load SpaceX Launch Data from API

In [53]:
import requests
import pandas as pd

In [54]:
# Step 1: Fetch data from API

url = "https://api.spacexdata.com/v4/launches"
response = requests.get(url)
data = response.json()

In [55]:
# Step 2: Convert to DataFrame

df = pd.DataFrame(data)

In [56]:
# Step 3: Select required columns

df = df[['name', 'date_utc', 'success', 'details', 'rocket']]

In [57]:
# Step 4: Convert date_utc to datetime

df['date_utc'] = pd.to_datetime(df['date_utc'])

In [58]:
# Step 5: Extract year

df['year'] = df['date_utc'].dt.year

# Step 6: Preview data

# print(df.head())
df.head()

,name,date_utc,success,details,rocket,year
0,FalconSat,2006-03-24 22:30:00+00:00,False,Engine failure at 33 seconds and loss of vehicle,5e9d0d95eda69955f709d1eb,2006
1,DemoSat,2007-03-21 01:10:00+00:00,False,Successful first stage burn and transition to ...,5e9d0d95eda69955f709d1eb,2007
2,Trailblazer,2008-08-03 03:34:00+00:00,False,Residual stage 1 thrust led to collision betwe...,5e9d0d95eda69955f709d1eb,2008
3,RatSat,2008-09-28 23:15:00+00:00,True,Ratsat was carried to orbit on the first succe...,5e9d0d95eda69955f709d1eb,2008
4,RazakSat,2009-07-13 03:35:00+00:00,True,None,5e9d0d95eda69955f709d1eb,2009


## Step 2: Load Rocket Metadata

In [59]:
# Step 1: Fetch rocket data

url = "https://api.spacexdata.com/v4/rockets"
response = requests.get(url)
rocket_data = response.json()

In [60]:
# Step 2: Convert to DataFrame

rocket_df = pd.DataFrame(rocket_data)

In [61]:
# Step 3: Select required columns

rocket_df = rocket_df[['id', 'name', 'type', 'active', 'stages']]

In [62]:
# Step 4: Rename 'id' to match launch dataset key

rocket_df = rocket_df.rename(columns={'id': 'rocket'})

# Step 5: Preview data

# print(rocket_df.head())
df.head()

,name,date_utc,success,details,rocket,year
0,FalconSat,2006-03-24 22:30:00+00:00,False,Engine failure at 33 seconds and loss of vehicle,5e9d0d95eda69955f709d1eb,2006
1,DemoSat,2007-03-21 01:10:00+00:00,False,Successful first stage burn and transition to ...,5e9d0d95eda69955f709d1eb,2007
2,Trailblazer,2008-08-03 03:34:00+00:00,False,Residual stage 1 thrust led to collision betwe...,5e9d0d95eda69955f709d1eb,2008
3,RatSat,2008-09-28 23:15:00+00:00,True,Ratsat was carried to orbit on the first succe...,5e9d0d95eda69955f709d1eb,2008
4,RazakSat,2009-07-13 03:35:00+00:00,True,None,5e9d0d95eda69955f709d1eb,2009


## Step 3: Merge Launch and Rocket Data

In [63]:
# Merge launch data with rocket metadata

merged_df = pd.merge(df, rocket_df, on='rocket', how='left')


In [64]:
# Preview merged data

# print(merged_df.head())
merged_df.head()

,name_x,date_utc,success,details,rocket,year,name_y,type,active,stages
0,FalconSat,2006-03-24 22:30:00+00:00,False,Engine failure at 33 seconds and loss of vehicle,5e9d0d95eda69955f709d1eb,2006,Falcon 1,rocket,False,2
1,DemoSat,2007-03-21 01:10:00+00:00,False,Successful first stage burn and transition to ...,5e9d0d95eda69955f709d1eb,2007,Falcon 1,rocket,False,2
2,Trailblazer,2008-08-03 03:34:00+00:00,False,Residual stage 1 thrust led to collision betwe...,5e9d0d95eda69955f709d1eb,2008,Falcon 1,rocket,False,2
3,RatSat,2008-09-28 23:15:00+00:00,True,Ratsat was carried to orbit on the first succe...,5e9d0d95eda69955f709d1eb,2008,Falcon 1,rocket,False,2
4,RazakSat,2009-07-13 03:35:00+00:00,True,None,5e9d0d95eda69955f709d1eb,2009,Falcon 1,rocket,False,2


## Step 4: Add Simulated Country Information

In [65]:
# Define country list

countries = ['USA', 'Russia', 'India', 'China', 'France']

In [66]:
import numpy as np

# Assign random country to each row

merged_df['country'] = np.random.choice(countries, size=len(merged_df))

In [67]:
# # Preview

# print(merged_df[['name_x', 'name_y', 'country']].head())
merged_df[['name_x', 'name_y', 'country']].head()

,name_x,name_y,country
0,FalconSat,Falcon 1,Russia
1,DemoSat,Falcon 1,USA
2,Trailblazer,Falcon 1,USA
3,RatSat,Falcon 1,France
4,RazakSat,Falcon 1,China


## Step 5: Store Merged Data in SQLite3

In [68]:
import sqlite3

# Step 1: Create connection (this creates a file if it doesn't exist)

conn = sqlite3.connect("spacex.db")

In [69]:
# Step 2: Store DataFrame into SQLite table

merged_df.to_sql("launches", conn, if_exists="replace", index=False)

205

In [74]:
# Step 3: Verify table creation

cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

# print(cursor.fetchall())
cursor.fetchall()

[('launches',)]

In [75]:
# Step 4: Preview data from table

query = "SELECT * FROM launches LIMIT 5;"
preview = pd.read_sql_query(query, conn)

# print(preview)
preview

,name_x,date_utc,success,details,rocket,year,name_y,type,active,stages,country
0,FalconSat,2006-03-24 22:30:00+00:00,0,Engine failure at 33 seconds and loss of vehicle,5e9d0d95eda69955f709d1eb,2006,Falcon 1,rocket,0,2,Russia
1,DemoSat,2007-03-21 01:10:00+00:00,0,Successful first stage burn and transition to ...,5e9d0d95eda69955f709d1eb,2007,Falcon 1,rocket,0,2,USA
2,Trailblazer,2008-08-03 03:34:00+00:00,0,Residual stage 1 thrust led to collision betwe...,5e9d0d95eda69955f709d1eb,2008,Falcon 1,rocket,0,2,USA
3,RatSat,2008-09-28 23:15:00+00:00,1,Ratsat was carried to orbit on the first succe...,5e9d0d95eda69955f709d1eb,2008,Falcon 1,rocket,0,2,France
4,RazakSat,2009-07-13 03:35:00+00:00,1,None,5e9d0d95eda69955f709d1eb,2009,Falcon 1,rocket,0,2,China


In [76]:
# Close connection

conn.close()

## Step 6: Run SQL Queries on the Data to analyze

In [77]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("spacex.db")

### 1. Launches by Country

In [78]:
query1 = """
SELECT country, COUNT(*) AS total_launches
FROM launches
GROUP BY country
ORDER BY total_launches DESC;
"""

df1 = pd.read_sql_query(query1, conn)

# print(df1)
df1

,country,total_launches
0,France,44
1,India,42
2,China,41
3,Russia,40
4,USA,38


### 2. Which year had the highest number of launches?


In [79]:
query2 = """
SELECT year, COUNT(*) AS total_launches
FROM launches
GROUP BY year
ORDER BY total_launches DESC
LIMIT 1;
"""

df2 = pd.read_sql_query(query2, conn)

# print(df2)
df2

,year,total_launches
0,2022,62


### 3. Top 5 Missions by Launch Count

In [80]:
query3 = """
SELECT name_x AS launch_name, COUNT(*) AS launch_count
FROM launches
GROUP BY name_x
ORDER BY launch_count DESC
LIMIT 5;
"""

df3 = pd.read_sql_query(query3, conn)

# print(df3)
df3

,launch_name,launch_count
0,ispace Mission 1 & Rashid,1
1,ZUMA,1
2,WorldView Legion 1 & 2,1
3,Viasat-3 & Arcturus,1
4,USSF-44,1
